# V9 CIFAR-10 Generation T4x2 1k Hardened

`NO_FAKE_RESULTS`  
`NO_REAL_EVIDENCE`  
`not paper evidence`  
`claim_allowed=false`

This notebook stops after generation packaging. It does not continue to feature extraction.

In [ ]:
import concurrent.futures, hashlib, json, os, shutil, stat, subprocess, sys, time, zipfile
from pathlib import Path
import torch
print('python', sys.version)
print('gpu_count', torch.cuda.device_count())
print('gpu_names', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print('disk', shutil.disk_usage('/kaggle/working'))
if torch.cuda.device_count() < 2:
    Path('/kaggle/working/generation_blocked_status.json').write_text(json.dumps({'status_code':'BLOCKED_CUDA_T4X2_NOT_AVAILABLE','claim_allowed':False}, indent=2))
    raise RuntimeError('T4x2 required')

In [ ]:
!pip -q install 'diffusers==0.34.0' 'transformers==4.53.2' 'accelerate==1.8.1' 'safetensors==0.5.3' 'pillow==11.2.1'
!python -m pip freeze > /kaggle/working/generation_dependency_freeze.txt
!python - <<'PY'
import diffusers, transformers, torch
print({'torch': torch.__version__, 'diffusers': diffusers.__version__, 'transformers': transformers.__version__})
PY

In [ ]:
# Checkpoint preflight cell: require imported preflight success or run the dedicated V9 preflight notebook first.
PREFLIGHT = Path('/kaggle/input/certgen-preflight/checkpoint_preflight_status.json')
if not PREFLIGHT.exists():
    Path('/kaggle/working/generation_blocked_status.json').write_text(json.dumps({'status_code':'BLOCKED_PREFLIGHT_MISSING','claim_allowed':False}, indent=2))
    raise FileNotFoundError('mandatory checkpoint preflight status is not attached')
preflight = json.loads(PREFLIGHT.read_text())
expected_revisions={'google/ddpm-cifar10-32':'267b167dc01f0e4e61923ea244e8b988f84deb80','FrankCCCCC/ddpm_ema_cifar10':'6aa387f240fbb00d0e003f93a3b994f56dd98dc2','FrankCCCCC/cfm-cifar10-32':'b3f30358497e11ce5011c00614c9b0521262f51c'}
preflight_results=preflight.get('results') or []
preflight_ok=preflight.get('claim_allowed') is False and preflight.get('status_code')=='PREFLIGHT_PASS' and len(preflight_results)==3 and all(r.get('status_code')=='PREFLIGHT_PASS' and expected_revisions.get(r.get('checkpoint_id'))==r.get('checkpoint_revision') for r in preflight_results)
if not preflight_ok:
    Path('/kaggle/working/generation_blocked_status.json').write_text(json.dumps({'status_code':'BLOCKED_PREFLIGHT_NOT_PASS','preflight':preflight,'claim_allowed':False}, indent=2))
    raise RuntimeError('checkpoint preflight did not pass exact revision contract')

In [ ]:
INPUT_ZIP=Path('/kaggle/input/certgen-generation/certgen_cifar10_generation_1k_input.zip')
WORK=Path('/kaggle/working/v9_generation_input')
if not INPUT_ZIP.exists():
    Path('/kaggle/working/generation_blocked_status.json').write_text(json.dumps({'status_code':'BLOCKED_INPUT_ZIP_MISSING','claim_allowed':False}, indent=2))
    raise FileNotFoundError(INPUT_ZIP)
input_hash=hashlib.sha256(INPUT_ZIP.read_bytes()).hexdigest()
marker=WORK/'.source_zip_sha256'
if WORK.exists():
    if not marker.is_file() or marker.read_text().strip()!=input_hash: raise RuntimeError('existing input extraction does not match attached ZIP')
else:
    WORK.mkdir(parents=True, exist_ok=False)
    with zipfile.ZipFile(INPUT_ZIP) as archive:
        infos=archive.infolist(); total=sum(i.file_size for i in infos)
        if len(infos)>10000 or total>2*1024**3 or archive.testzip() is not None: raise RuntimeError('input ZIP failed size/CRC limits')
        seen=set()
        for info in infos:
            parts=Path(info.filename).parts; mode=(info.external_attr>>16)&0xFFFF
            if info.filename.startswith('/') or '..' in parts or '\\' in info.filename or info.filename.casefold() in seen or (mode and stat.S_ISLNK(mode)): raise RuntimeError(f'unsafe input ZIP member: {info.filename}')
            seen.add(info.filename.casefold())
            target=WORK/info.filename
            if info.is_dir(): target.mkdir(parents=True, exist_ok=True); continue
            target.parent.mkdir(parents=True, exist_ok=True); target.write_bytes(archive.read(info))
    marker.write_text(input_hash)
sys.path.insert(0,str(WORK/'repo')); os.environ['PYTHONPATH']=str(WORK/'repo')
checkpoints=json.loads((WORK/'config/checkpoints.json').read_text())['checkpoints']
assert len(checkpoints)==3
assert {c.get('checkpoint_id'):c.get('revision') for c in checkpoints}==expected_revisions
SAMPLE_ROOT=Path('/kaggle/working/samples'); MANIFEST_ROOT=Path('/kaggle/working/manifests'); LOG_ROOT=Path('/kaggle/working/logs'); STATUS_ROOT=Path('/kaggle/working/status')
for p in [SAMPLE_ROOT, MANIFEST_ROOT, LOG_ROOT, STATUS_ROOT]: p.mkdir(parents=True, exist_ok=True)

In [ ]:
def manifest_is_complete(manifest, checkpoint_id, revision, seed_start, seed_end):
    try: rows=[json.loads(line) for line in manifest.read_text().splitlines() if line.strip()]
    except Exception: return False
    return len(rows)==seed_end-seed_start and {r.get('seed') for r in rows}==set(range(seed_start,seed_end)) and all(r.get('checkpoint_id')==checkpoint_id and r.get('checkpoint_revision')==revision and r.get('claim_allowed') is False and Path(r.get('image_path','')).is_file() and hashlib.sha256(Path(r['image_path']).read_bytes()).hexdigest()==r.get('image_hash') for r in rows)
def run_shard(checkpoint_id, revision, short_id, gpu, seed_start, seed_end):
    manifest = MANIFEST_ROOT / f'{short_id}_gpu{gpu}.jsonl'
    status = STATUS_ROOT / f'{short_id}_gpu{gpu}_status.json'
    if status.exists() and json.loads(status.read_text()).get('status_code') == 'SHARD_COMPLETE' and manifest_is_complete(manifest, checkpoint_id, revision, seed_start, seed_end):
        return json.loads(status.read_text())
    args=[sys.executable,'-m','certgen.generation.generate_cifar10_diffusers','--checkpoint-id',checkpoint_id,'--seed-start',str(seed_start),'--seed-end',str(seed_end),'--num-samples',str(seed_end-seed_start),'--out-dir',f'/kaggle/working/samples/{short_id}/gpu{gpu}','--manifest-out',str(manifest),'--device','cuda','--batch-size','32','--resume','--execute']
    command=f"CUDA_VISIBLE_DEVICES={gpu} "+' '.join(args)
    start=time.time(); code=1
    for attempt in range(2):
        env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        with (LOG_ROOT/f'{short_id}_gpu{gpu}.log').open('a') as log: code=subprocess.run(args, env=env, stdout=log, stderr=subprocess.STDOUT).returncode
        if code==0: break
        time.sleep(5)
    complete=code==0 and manifest_is_complete(manifest, checkpoint_id, revision, seed_start, seed_end)
    payload={'checkpoint_id':checkpoint_id,'checkpoint_revision':revision,'short_id':short_id,'gpu':gpu,'seed_start':seed_start,'seed_end':seed_end,'status_code':'SHARD_COMPLETE' if complete else 'SHARD_FAILED','manifest_sha256':hashlib.sha256(manifest.read_bytes()).hexdigest() if manifest.exists() else None,'wall_time_seconds':time.time()-start,'resume_supported':True,'failed_shard_rerun':command,'claim_allowed':False}
    temporary=status.with_suffix('.json.tmp'); temporary.write_text(json.dumps(payload, indent=2)); os.replace(temporary,status)
    return payload
all_status=[]
for item in checkpoints:
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
        futures=[pool.submit(run_shard,item['checkpoint_id'],item['revision'],item['short_id'],0,0,500),pool.submit(run_shard,item['checkpoint_id'],item['revision'],item['short_id'],1,500,1000)]
        all_status.extend(future.result() for future in futures)
if any(s['status_code']!='SHARD_COMPLETE' for s in all_status):
    Path('/kaggle/working/generation_blocked_status.json').write_text(json.dumps({'status_code':'BLOCKED_PARTIAL_GENERATION_FAILURE','shards':all_status,'claim_allowed':False}, indent=2))
    raise RuntimeError('partial generation failure')

In [ ]:
manifests=['/kaggle/working/manifests/google_ddpm_gpu0.jsonl','/kaggle/working/manifests/google_ddpm_gpu1.jsonl','/kaggle/working/manifests/frank_ddpm_ema_gpu0.jsonl','/kaggle/working/manifests/frank_ddpm_ema_gpu1.jsonl','/kaggle/working/manifests/frank_cfm_gpu0.jsonl','/kaggle/working/manifests/frank_cfm_gpu1.jsonl']
merge_args=[sys.executable,'-m','certgen.generation.merge_sample_manifests']+sum((['--manifest',item] for item in manifests),[])+['--out-manifest','/kaggle/working/manifests/cifar10_r1_generated_pilot_1000.jsonl','--out-summary','/kaggle/working/status/generated_manifest_merge_summary.json','--check-image-hashes']
subprocess.run(merge_args, check=True)
subprocess.run([sys.executable,'-m','certgen.generation.validate_cifar10_generated_pilot','--manifest-dir','/kaggle/working/manifests','--out-manifest','/kaggle/working/manifests/cifar10_r1_generated_pilot_1000.validated.jsonl','--out-summary','/kaggle/working/status/generation_status.json','--expected-count-per-model','1000','--check-image-hashes'], check=True)
status=json.loads(Path('/kaggle/working/status/generation_status.json').read_text())
assert status['claim_allowed'] is False
# Duplicate ids, seeds, paths, or hashes are hard validation failures.
if not status.get('passed'):
    Path('/kaggle/working/generation_blocked_status.json').write_text(json.dumps({'status_code':'BLOCKED_GENERATED_MANIFEST_INVALID','status':status,'claim_allowed':False}, indent=2))
    raise RuntimeError(status)

In [ ]:
integrity=[]
for path in Path('/kaggle/working').rglob('*'):
    if path.is_file() and ('samples' in path.parts or 'manifests' in path.parts or 'status' in path.parts or 'logs' in path.parts):
        integrity.append({'path':str(path.relative_to('/kaggle/working')),'size':path.stat().st_size,'sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
integrity.append({'path':'logs/generation_dependency_freeze.txt','size':Path('/kaggle/working/generation_dependency_freeze.txt').stat().st_size,'sha256':hashlib.sha256(Path('/kaggle/working/generation_dependency_freeze.txt').read_bytes()).hexdigest()})
Path('/kaggle/working/status/output_zip_integrity_manifest.json').write_text(json.dumps({'files':integrity,'claim_allowed':False}, indent=2))
ZIP=Path('/kaggle/working/certgen_cifar10_generation_outputs_v9_1k.zip')
if ZIP.exists(): raise FileExistsError(f'refusing to overwrite {ZIP}')
with zipfile.ZipFile(ZIP,'x',compression=zipfile.ZIP_DEFLATED) as archive:
    for root_name in ['samples','manifests','logs','status']:
        for path in sorted((Path('/kaggle/working')/root_name).rglob('*')):
            if path.is_file(): archive.write(path,path.relative_to('/kaggle/working'))
    archive.write('/kaggle/working/generation_dependency_freeze.txt','logs/generation_dependency_freeze.txt')
print('Copy back /kaggle/working/certgen_cifar10_generation_outputs_v9_1k.zip to data/kaggle_outputs/')
print('Then run commands/v9_cpu_execution/03_import_generation_zip_v9.sh')